# Structure-Aware Semantic-Guided Thermal to Visible Image Translation
## Training Pipeline on LLVIP Dataset (Kaggle GPU Environment)

This notebook runs the complete 3-Stage training framework:
1. **Stage 1**: Pre-train Structure Extraction Network (Module 3A)
2. **Stage 2**: Train Feature Fusion (Module 4) + Appearance Generator (Module 5)
3. **Stage 3**: End-to-End Joint Fine-Tuning
4. **Evaluation**: PSNR, SSIM, and Visual Comparison Grid Generation

In [ ]:
# Step 1: Verify GPU & LLVIP Dataset Path
import os
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# Check LLVIP Dataset presence on Kaggle
dataset_root = "/kaggle/input/llvip-dataset/LLVIP"
if not os.path.exists(dataset_root):
    print(f"⚠️ Warning: LLVIP dataset not found at {dataset_root}.")
    print("Please click '+ Add Data' in the right panel and attach the LLVIP dataset.")
else:
    print(f"✅ LLVIP Dataset detected at: {dataset_root}")
    print("Thermal train count:", len(os.listdir(os.path.join(dataset_root, 'thermal', 'train'))))
    print("Visible train count:", len(os.listdir(os.path.join(dataset_root, 'visible', 'train'))))

--- 
## Stage 1: Pre-train Structure Extraction Network (Module 3A)
Extracts edge maps, contours, and high-frequency structural features from thermal inputs.

In [ ]:
!python train_stage1_structure.py

--- 
## Stage 2: Train Feature Fusion (4) & Appearance Generator (5)
Fuses structural (3A) and semantic (3B) features to synthesize realistic visible RGB images.

In [ ]:
!python train_stage2_generator.py

--- 
## Stage 3: End-to-End Joint Fine-Tuning
Unfreezes all modules (3A, 3B, 4, 5) and optimizes the full system jointly.

In [ ]:
!python train_stage3_end2end.py

--- 
## Stage 4: Quantitative & Qualitative Evaluation
Computes PSNR and SSIM metrics and displays side-by-side comparison images.

In [ ]:
!python evaluate.py

# Display sample output grid inline in notebook
import glob
from PIL import Image
import matplotlib.pyplot as plt

results = sorted(glob.glob("./results/*.jpg")) + sorted(glob.glob("./results/*.png"))
if results:
    print(f"Total Generated Sample Grids: {len(results)}")
    fig, axes = plt.subplots(min(3, len(results)), 1, figsize=(15, 10))
    if len(results) == 1:
        axes = [axes]
    for i, img_path in enumerate(results[:3]):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f"Sample {i+1}: Thermal | Structure | Generated Visible | Ground Truth RGB | Confidence")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()